In [ ]:
!pip install -q faiss-cpu
!pip install -q sentence-transformers
!pip install -q pypdf
!pip install -q tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os
import json
import faiss
import numpy as np
from tqdm import tqdm
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer

# =====================================
# CONFIG
# =====================================

SOURCE_BASE = "/content/drive/MyDrive/CB/Nancy"
TARGET_BASE = "/content/drive/MyDrive/CB/Nancy_INDEX"

CHUNK_SIZE = 500
CHUNK_OVERLAP = 100
EMBED_MODEL = "all-MiniLM-L6-v2"

print("Loading embedding model...")
embed_model = SentenceTransformer(EMBED_MODEL)

# =====================================
# CHUNK FUNCTION
# =====================================

def chunk_text(text, chunk_size=500, overlap=100):
    words = text.split()
    chunks = []
    step = chunk_size - overlap

    for i in range(0, len(words), step):
        chunk = " ".join(words[i:i + chunk_size])
        if len(chunk.strip()) > 50:
            chunks.append(chunk)

    return chunks

# =====================================
# PROCESS ONE FOLDER
# =====================================

def process_folder(source_folder):

    relative_path = os.path.relpath(source_folder, SOURCE_BASE)
    target_folder = os.path.join(TARGET_BASE, relative_path)

    pdf_files = [f for f in os.listdir(source_folder) if f.endswith(".pdf")]

    if not pdf_files:
        return

    print(f"\nProcessing: {relative_path}")

    os.makedirs(target_folder, exist_ok=True)

    all_chunks = []
    metadata = []

    for file in pdf_files:
        pdf_path = os.path.join(source_folder, file)

        try:
            reader = PdfReader(pdf_path)
            text = ""

            for page in reader.pages:
                extracted = page.extract_text()
                if extracted:
                    text += extracted

            chunks = chunk_text(text, CHUNK_SIZE, CHUNK_OVERLAP)

            for chunk in chunks:
                all_chunks.append(chunk)
                metadata.append({
                    "source_file": file,
                    "folder": relative_path
                })

        except Exception as e:
            print(f"Error reading {file}: {e}")

    if len(all_chunks) == 0:
        print("No valid content found.")
        return

    print("Creating embeddings...")

    embeddings = embed_model.encode(
        all_chunks,
        show_progress_bar=True,
        convert_to_numpy=True
    ).astype("float32")

    dimension = embeddings.shape[1]
    index = faiss.IndexFlatL2(dimension)
    index.add(embeddings)

    # ==========================
    # SAVE FILES SEPARATELY
    # ==========================

    # Save FAISS index
    faiss.write_index(index, os.path.join(target_folder, "index.faiss"))

    # Save chunks.json
    with open(os.path.join(target_folder, "chunks.json"), "w") as f:
        json.dump(all_chunks, f)

    # Save metadata.json
    with open(os.path.join(target_folder, "metadata.json"), "w") as f:
        json.dump(metadata, f)

    print("Saved index.faiss, chunks.json, metadata.json")

# =====================================
# PROCESS ALL SUBFOLDERS
# =====================================

for root, dirs, files in os.walk(SOURCE_BASE):
    process_folder(root)

print("\n✅ All folders processed successfully.")

Loading embedding model...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

Saved index.faiss, chunks.json, metadata.json

Processing: Industrial and Organizational Psychology
Creating embeddings...


Batches:   0%|          | 0/30 [00:00<?, ?it/s]

Saved index.faiss, chunks.json, metadata.json

Processing: Health Psychology and Addiction
Creating embeddings...


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Saved index.faiss, chunks.json, metadata.json

✅ All folders processed successfully.


In [ ]:
import os, json

BASE = "/content/drive/Shared drives/CB/Nancy_INDEX"

for root, dirs, files in os.walk(BASE):
    if "chunks.json" in files:
        path = os.path.join(root, "chunks.json")
        with open(path) as f:
            chunks = json.load(f)
        print(root, "→", len(chunks), "chunks")

In [ ]:
!pip install pyngrok

In [ ]:
!pip install streamlit pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 121.2 MB/s eta 0:00:00


In [ ]:
!streamlit run app.py &>/dev/null &

In [ ]:
!pkill -f streamlit

In [ ]:
!pip install sentence-transformers faiss-cpu

In [ ]:
%%writefile app.py

import streamlit as st
import faiss
import json
import numpy as np
import torch
import re
import os

from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

# =====================================================
# PAGE SETTINGS
# =====================================================
st.set_page_config(page_title="🧠 Personal Mental Health Chatbot")
st.title("🧠 Personal Mental Health Chatbot")

DEVICE = "cpu"
RELEVANCE_THRESHOLD = 0.55
TOP_K = 3

# =====================================================
# BASE INDEX DIRECTORY
# =====================================================
BASE_INDEX_PATH = "/content/drive/MyDrive/CB/Nancy_INDEX"
if not os.path.exists(BASE_INDEX_PATH):
    st.error("Nancy_INDEX folder not found.")
    st.stop()

# =====================================================
# CATEGORY ROUTER
# =====================================================
CATEGORY_KEYWORDS = {

    "Questionnaires/depression":[
        "depression","sad","hopeless","worthless","empty","lonely","no motivation"
    ],

    "Questionnaires/anxiety":[
        "anxiety","panic","fear","nervous","overthinking","social anxiety","worry"
    ],

    "Questionnaires/stress and burnout":[
        "stress","burnout","pressure","overwhelmed","tired","exhausted","exam"
    ],

    "Questionnaires/ptsd":[
        "ptsd","trauma","flashback","nightmares","traumatic memory"
    ],

    "Questionnaires/substance abuse":[
        "substance","drug","addiction","alcohol","smoking","dependency"
    ],

    "Questionnaires/suicidal ideation":[
        "suicide","kill myself","end my life","self harm","want to die"
    ],

    "Questionnaires/emotional regulation":[
        "emotional control","control my emotions","anger issues","mood swings"
    ],

    "Questionnaires/marital life":[
        "marriage problem","husband","wife","relationship","marital conflict"
    ],

    "Questionnaires/well being":[
        "wellbeing","life satisfaction","happiness","positive life"
    ],

    "Questionnaires/workplace":[
        "boss","job stress","workplace","office stress","manager problem"
    ],

    "Questionnaires/GENERAL HEALTH":[
        "sleep","health","fatigue","energy","daily functioning"
    ],

    "Health Psychology and Addiction":[
        "addiction","health behavior","substance dependency"
    ],

    "Industrial and Organizational Psychology":[
        "organizational","work productivity","employee stress","work culture"
    ],

    "Counselling and Therapies":[
        "therapy","counselling","cbt","treatment","psychotherapy"
    ],

    "Freud Psychology and Defense mechanisms":[
        "defense mechanism","repression","projection","freud"
    ],

    "Abnormal Psychology & ICD 11":[
        "mental disorder","psychological disorder","diagnosis","icd"
    ],

    "Miscellaneous":[
        "mental health","psychology","emotional problem"
    ]
}

def detect_category(query):
    q=query.lower()
    best_match=None
    best_score=0

    for category,keywords in CATEGORY_KEYWORDS.items():
        score=sum(1 for k in keywords if k in q)

        if score>best_score:
            best_score=score
            best_match=category

    return best_match

# =====================================================
# LOAD MODELS
# =====================================================
@st.cache_resource
def load_embedding():
    return SentenceTransformer("all-MiniLM-L6-v2")

@st.cache_resource
def load_llm():

    model_name="Qwen/Qwen2.5-0.5B-Instruct"

    tokenizer=AutoTokenizer.from_pretrained(model_name)

    model=AutoModelForCausalLM.from_pretrained(
        model_name,
        torch_dtype=torch.float32
    )

    model.to(DEVICE)
    model.eval()

    return tokenizer,model

embedding_model=load_embedding()
tokenizer,model=load_llm()

# =====================================================
# LOAD INDEX
# =====================================================
@st.cache_resource
def load_category_index(category_path):

    full_path=os.path.join(BASE_INDEX_PATH,category_path)

    faiss_path=os.path.join(full_path,"index.faiss")
    chunks_path=os.path.join(full_path,"chunks.json")

    if not os.path.exists(faiss_path) or not os.path.exists(chunks_path):
        return None,None

    index=faiss.read_index(faiss_path)

    with open(chunks_path,"r",encoding="utf-8") as f:
        chunks=json.load(f)

    return index,chunks

def extract_text(chunk):

    if isinstance(chunk,str):
        return chunk

    if isinstance(chunk,dict) and "text" in chunk:
        return chunk["text"]

    return ""

# =====================================================
# RETRIEVAL
# =====================================================
def retrieve_best_chunk(query):

    category=detect_category(query)

    if not category:
        return None,None,0.0

    index,chunks=load_category_index(category)

    if index is None:
        return None,category,0.0

    query_embedding=embedding_model.encode(
        [query],
        normalize_embeddings=True
    ).astype("float32")

    D,I=index.search(query_embedding,TOP_K)

    selected_chunks=[
        extract_text(chunks[idx])
        for score,idx in zip(D[0],I[0])
        if idx>=0 and score>=RELEVANCE_THRESHOLD
    ]

    if not selected_chunks:
        return None,category,float(D[0][0])

    return "\n\n".join(selected_chunks),category,float(D[0][0])

# =====================================================
# CLEAN TEXT
# =====================================================
def clean_generated_text(text):

    text=re.sub(r'\*\*','',text)
    text=re.sub(r'\s+',' ',text)

    return text.strip()

# =====================================================
# SENTENCE SPLIT
# =====================================================
def split_sentences(text):

    sentences=re.split(r'(?<=[.!?])\s+',text)

    return [s.strip() for s in sentences if len(s.strip())>30]

# =====================================================
# BRIEF SUMMARY
# =====================================================
def generate_brief_summary(query):

    prompt=f"""
Summarize the emotional issue in the following question in 2 short sentences.

User question:
{query}

Summary:
"""

    inputs=tokenizer(prompt,return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs=model.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.3
        )

    generated_tokens=outputs[0][inputs["input_ids"].shape[-1]:]

    summary=tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    return clean_generated_text(summary)

# =====================================================
# MOTIVATION (IMPROVED NATURAL VERSION)
# =====================================================
def generate_motivation(query):

    prompt=f"""
You are a compassionate mental health guide.

Write a meaningful motivational reflection for someone facing the following emotional situation.

The reflection should feel natural, supportive, and deeply encouraging.
It can include a small inspiring thought, a life lesson, or a gentle perspective about overcoming struggles.

Important rules:
- Do NOT give instructions or numbered steps.
- Do NOT diagnose mental illness.
- Write in a warm and human tone.
- Focus on hope, resilience, and growth.

Emotional situation:
{query}

Motivational reflection:
"""

    inputs=tokenizer(prompt,return_tensors="pt").to(DEVICE)

    with torch.no_grad():

        outputs=model.generate(
            **inputs,
            max_new_tokens=180,
            temperature=0.75,
            top_p=0.92,
            repetition_penalty=1.15,
            do_sample=True
        )

    generated_tokens=outputs[0][inputs["input_ids"].shape[-1]:]

    text=tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    text=clean_generated_text(text)

    text=re.sub(r'["“”]',"",text)
    text=re.sub(r'\d+\)',"",text)
    text=re.sub(r'\s+',' ',text)

    sentences=re.split(r'(?<=[.!?])\s+',text)

    sentences=[s.strip() for s in sentences if len(s.strip())>20]

    sentences=sentences[:8]

    return "\n".join(sentences)

# =====================================================
# FORMAT STRUCTURED ANSWER
# =====================================================
def format_structured_answer(text,brief_summary,motivation):

    text=clean_generated_text(text)

    sentences=split_sentences(text)

    if len(sentences)<6:
        return text

    intro=sentences[0]
    explanation=" ".join(sentences[1:6])
    key_points=sentences[6:9] if len(sentences)>=9 else sentences[2:5]
    impact=sentences[9:12] if len(sentences)>=12 else sentences[-4:-1]
    conclusion=sentences[-1]

    formatted=f"""
**Introduction**

{intro}

**Brief Explanation**

{brief_summary}

**Explanation**

{explanation}

**Key Points**
"""

    for kp in key_points:
        formatted+=f"\n- {kp}"

    formatted+="\n\n**Impact and Coping Strategies**"

    for im in impact:
        formatted+=f"\n- {im}"

    formatted+=f"""

**Conclusion**

{conclusion}

**Motivation**

{motivation}
"""

    return formatted.strip()

# =====================================================
# GENERATION SETTINGS
# =====================================================
GEN_KWARGS=dict(
    max_new_tokens=420,
    temperature=0.4,
    top_p=0.9,
    repetition_penalty=1.2,
    do_sample=True
)

# =====================================================
# GENERATE ANSWER
# =====================================================
def generate_answer(query,context=None):

    system_message=(
        "You are a compassionate mental health support assistant.\n\n"
        "Rules:\n"
        "- Do not diagnose mental illness.\n"
        "- Explain emotional experiences clearly.\n"
        "- Use supportive language.\n"
        "- Provide helpful coping suggestions.\n"
    )

    if context:

        user_message=f"""
Reference material:
{context}

User question:
{query}

Explain the emotional situation clearly.
"""

    else:

        user_message=f"""
User question:
{query}

Explain the emotional challenge in supportive language.
"""

    messages=[
        {"role":"system","content":system_message},
        {"role":"user","content":user_message}
    ]

    prompt=tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs=tokenizer(prompt,return_tensors="pt").to(DEVICE)

    with torch.no_grad():
        outputs=model.generate(**inputs,**GEN_KWARGS)

    generated_tokens=outputs[0][inputs["input_ids"].shape[-1]:]

    response=tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    brief_summary=generate_brief_summary(query)

    motivation=generate_motivation(query)

    return format_structured_answer(response,brief_summary,motivation)

# =====================================================
# CHAT UI
# =====================================================
if "chat_history" not in st.session_state:
    st.session_state.chat_history=[]

user_input=st.chat_input("Ask about stress, anxiety, sleep, depression...")

if user_input:

    st.session_state.chat_history.append(
        {"role":"user","content":user_input}
    )

    with st.spinner("Thinking..."):

        chunk,category,score=retrieve_best_chunk(user_input)

        reply=generate_answer(
            user_input,
            chunk if chunk else None
        )

        source_type="📖 RAG Knowledge Base" if chunk else "🤖 Model Response"

    st.session_state.chat_history.append(
        {"role":"assistant","content":reply}
    )

    with st.expander("🔎 Transparency Details"):

        st.write("Detected Category:",category)

        st.write("Similarity Score:",round(score,4))

        if chunk:
            st.write("Retrieved Context Preview:")
            st.write(chunk[:500])
        else:
            st.write("No relevant chunk found.")

        st.write("Answer Source:",source_type)

for msg in st.session_state.chat_history:

    with st.chat_message("user" if msg["role"]=="user" else "assistant"):

        st.markdown(
            f"<span style='font-size:16px'>{msg['content']}</span>",
            unsafe_allow_html=True
        )

Writing app.py


In [ ]:

from pyngrok import ngrok

ngrok.set_auth_token("39yHDphsAXGu68A4ZJHuvquoswI_6Jx1gd8aZ9ySJwXs1bgBY")

In [ ]:
from pyngrok import ngrok

ngrok.kill()

In [ ]:
public_url = ngrok.connect(8501)
print(public_url)

NgrokTunnel: "https://keila-unkidnapped-tien.ngrok-free.dev" -> "http://localhost:8501"


In [ ]:

!streamlit run app.py --server.port 8501 --server.address 0.0.0.0


  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.186.21.16:8501

Loading weights: 100% 103/103 [00:00<00:00, 2079.03it/s, Materializing param=pooler.dense.weight]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
config.json: 100% 659/659 [00:00<00:00, 2.73MB/s]
tokenizer_config.json: 7.30kB [00:00, 18.6MB/s]
vocab.json: 2.78MB [00:00, 62.0MB/s]
merges.txt: 1.67MB [00:00, 108MB/s]
tokenizer.json: 7.03MB [00:00, 137MB/s]
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors: 100% 988M/988M [00:09<00:00, 103MB/s]
Loading weights: 100% 290/290 [00:02<00:00, 107.90it/s, Materializing param=model.norm.weight]